# 14. Skater WAR history and goalie WAR v0

Phase 0 of Forecast. Projections need career history and the whole roster, so this notebook does two things:

- **A. Skater WAR back to 2008-09.** The same v0.4 model, no math changes, run on MoneyPuck's combined 2008-2024 file plus 2025-26: 18 seasons.
- **B. Goalie WAR v0.** Goals saved above expected (GSAx) above a replacement-level backup, in wins. Checked the same way as v0.4: does team WAR match team goal differential better once goalies are added?

The goalie code lives in `src/goalie_war.py`. Nothing on the cards changes.

Data: MoneyPuck.com season-level skater and goalie files (`data/skaters_2008_to_2024.csv`, `data/skaters25-26.csv`, `data/goalies_2008_to_2024.csv`, `data/goalies25-26.csv`). season = start year (2025 = 2025-26).

## 1. Setup

In [1]:
import sys
sys.path.append("../src")

import pandas as pd
from pathlib import Path

from war_model import run_season, V04_PARAMS
from goalie_war import (load_goalies, replacement_rates, goalie_war, team_check, franchise_check, season_pairs,
                        repeatability, season_label, FRANCHISE, GOALS_PER_WIN, MIN_REPEAT_TOI, MIN_PK_TOI)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

DATA = Path("../data")
OUT = DATA                      # where the two new tables are saved

skaters_raw = pd.concat([pd.read_csv(DATA / "skaters_2008_to_2024.csv"), pd.read_csv(DATA / "skaters25-26.csv")],
                        ignore_index=True)
goalies_raw = pd.concat([pd.read_csv(DATA / "goalies_2008_to_2024.csv"), pd.read_csv(DATA / "goalies25-26.csv")],
                        ignore_index=True)
SEASONS = [int(s) for s in sorted(skaters_raw["season"].unique())]
assert SEASONS == list(range(2008, 2026))
assert [int(s) for s in sorted(goalies_raw["season"].unique())] == SEASONS

print(f"Skater rows: {len(skaters_raw):,}   Goalie rows: {len(goalies_raw):,}")
print(f"Seasons: {season_label(SEASONS[0])} to {season_label(SEASONS[-1])} ({len(SEASONS)})")
print("Goals per win (from war_model, used for goalies too):", GOALS_PER_WIN)

Skater rows: 81,355   Goalie rows: 8,510
Seasons: 2008-09 to 2025-26 (18)
Goals per win (from war_model, used for goalies too): 6.0


# Part A. Skater WAR, 2008-09 to 2025-26

## 2. Run v0.4 on all 18 seasons
Each season is run on its own, exactly as before. `run_season` now also accepts one season's rows from the combined file.

In [2]:
war = pd.concat([run_season(skaters_raw[skaters_raw["season"] == s], V04_PARAMS) for s in SEASONS], ignore_index=True)
print(war.groupby("season").size().rename(season_label).rename("skaters").to_frame().T.to_string())

season   2008-09  2009-10  2010-11  2011-12  2012-13  2013-14  2014-15  2015-16  2016-17  2017-18  2018-19  2019-20  2020-21  2021-22  2022-23  2023-24  2024-25  2025-26
skaters      885      879      891      894      839      886      882      898      887      890      906      883      913     1003      951      924      920      940


## 3. Regression check: 2021-22 to 2025-26 must not change
Two checks:
- **(a)** For 2021-22 to 2024-25, the combined file must give **exactly** the same table as the per-season files (run here, in memory).
- **(b)** It must also match the saved `war_by_season_v04.csv` (notebook 06). Names, teams, games and ranks must be identical. Decimals must agree to within 1e-9: saving a number to CSV and reading it back can change its 16th digit (about 1e-13). Rerunning 2025-26 from its own unchanged file shows the same tiny difference, so it isn't the data.

In [3]:
KEEP = list(pd.read_csv(DATA / "war_by_season_v04.csv", nrows=0).columns)     # same columns as the v0.4 table
SORT = ["season", "playerId"]

def tidy(t):
    return t[KEEP].sort_values(SORT).reset_index(drop=True)

# (a) combined file vs. per-season files: identical, bit for bit
per_season = pd.concat([run_season(DATA / f"skaters{str(s)[-2:]}-{str(s + 1)[-2:]}.csv", V04_PARAMS)
                        for s in range(2021, 2025)])
assert tidy(war[war["season"].between(2021, 2024)]).equals(tidy(per_season))
print(f"(a) Combined file = per-season files for 2021-22 to 2024-25: {len(per_season):,} player-seasons, identical")

# (b) vs. the saved v0.4 table (2021-22 to 2025-26)
old = tidy(pd.read_csv(DATA / "war_by_season_v04.csv"))
new = tidy(war[war["season"] >= 2021])
assert new[SORT].equals(old[SORT]), "different player-seasons"
decimals = [c for c in KEEP if pd.api.types.is_float_dtype(old[c])]
exact = [c for c in KEEP if c not in decimals]
assert new[exact].equals(old[exact]), "text or whole-number columns differ"
largest = (new[decimals] - old[decimals]).abs().max().max()
assert largest < 1e-9
print(f"(b) Saved war_by_season_v04.csv: {len(new):,} player-seasons match. {len(exact)} text/whole-number columns "
      f"identical (incl. war_rank); largest difference in {len(decimals)} decimal columns: {largest:.1e}")

(a) Combined file = per-season files for 2021-22 to 2024-25: 3,798 player-seasons, identical
(b) Saved war_by_season_v04.csv: 4,738 player-seasons match. 8 text/whole-number columns identical (incl. war_rank); largest difference in 15 decimal columns: 2.3e-13


## 4. Games per season
Each team's most-played skater (traded players can show 83 to 85, because their games with both teams count). Three seasons were shortened. WAR is a season total, so WAR in those seasons is smaller: compare them per game, not raw.

In [4]:
SHORTENED = {2012: "lockout, 48 games", 2019: "stopped in March (COVID), 68 to 71 games", 2020: "56 games, in-division"}

alls = skaters_raw[skaters_raw["situation"] == "all"]
team_gp = alls.groupby(["season", "team"])["games_played"].max()
games = team_gp.groupby("season").agg(["median", "min", "max"]).astype(int)
games.columns = ["games (median team)", "min", "max"]
games["shortened"] = games.index.map(lambda s: SHORTENED.get(s, ""))
games.index = games.index.map(season_label)

assert games.loc["2012-13", "games (median team)"] == 48 and games.loc["2020-21", "games (median team)"] == 56
assert games.loc["2019-20", "games (median team)"] < 72
assert (games.drop(["2012-13", "2019-20", "2020-21"])["games (median team)"] == 82).all()
games

,games (median team),min,max,shortened
season,,,,
2008-09,82,74,83,
2009-10,82,81,83,
2010-11,82,80,83,
2011-12,82,79,84,
2012-13,48,47,48,"lockout, 48 games"
2013-14,82,80,83,
2014-15,82,81,84,
2015-16,82,81,84,
2016-17,82,80,82,


## 5. Top 5 WAR per season, 2008-09 to 2020-21
A sanity check against hockey history: the leaders should be the players widely seen as the best that year.

In [5]:
for s in range(2008, 2021):
    top = war[war["season"] == s].nlargest(5, "war")[["name", "team", "position", "war"]].round(1)
    print(season_label(s) + (f"   (shortened: {SHORTENED[s]})" if s in SHORTENED else ""))
    print(top.to_string(index=False))
    print()

2008-09
           name team position  war
   Marian Hossa  DET        R  4.9
  Alex Ovechkin  WSH        L  4.9
    Zach Parise  N.J        L  4.7
  Pavel Datsyuk  DET        C  4.7
Alexander Semin  WSH        R  4.3

2009-10
           name team position  war
  Alex Ovechkin  WSH        L  6.9
  Sidney Crosby  PIT        C  5.9
 Marian Gaborik  NYR        R  5.0
 Ilya Kovalchuk  N.J        R  4.7
Alexander Semin  WSH        R  4.4

2010-11
          name team position  war
  Daniel Sedin  VAN        L  5.8
   Corey Perry  ANA        R  4.9
Steven Stamkos  T.B        C  4.5
Jonathan Toews  CHI        C  4.4
   Ryan Kesler  VAN        C  4.3

2011-12
          name team position  war
 Evgeni Malkin  PIT        C  6.8
Steven Stamkos  T.B        C  6.4
 Jordan Eberle  EDM        R  4.6
  John Tavares  NYI        C  4.2
 Logan Couture  S.J        C  4.0

2012-13   (shortened: lockout, 48 games)
          name team position  war
 Alex Ovechkin  WSH        L  4.0
   Nazem Kadri  TOR        

## 6. Save
`war_by_season_2008_2025.csv`: every skater, all 18 seasons, same columns as `war_by_season_v04.csv` (which is left as is).

In [6]:
war[KEEP].to_csv(OUT / "war_by_season_2008_2025.csv", index=False)
print(f"Saved {len(war):,} player-seasons ({war['season'].nunique()} seasons) to war_by_season_2008_2025.csv")

Saved 16,271 player-seasons (18 seasons) to war_by_season_2008_2025.csv


# Part B. Goalie WAR v0

## 7. Goals saved above expected, above replacement
- **GSAx** = expected goals against minus goals against, all situations. MoneyPuck's regular expected goals, the same ones the skater model uses.
- **Replacement level:** goalies outside each team's top 2 by ice time (like the skaters' top 13 forwards and 7 defensemen).
- **One season's backup pool is small (23 to 55 goalies), so its rate jumps around.** Instead, the backups' gap below the league average is pooled over all 18 seasons, and each season's replacement rate is that season's league average plus that gap. Both versions are shown below. The per-season version swings total goalie WAR from about 27 to 305 a season; the steady version keeps it near 113 to 121 in a full season.
- **GAR** = GSAx minus the replacement rate x minutes / 60. **WAR** = GAR / goals per win (same constant as skaters).

In [7]:
goalies = load_goalies(goalies_raw)
rates, gap = replacement_rates(goalies)
gw = goalie_war(goalies, rates)                                           # steady gap: used
gw_per_season = goalie_war(goalies, rates, rate_col="rate_per_season")    # per-season pool: comparison only

print(f"{len(goalies):,} goalie-seasons. Backups allow {-gap:.3f} more goals per 60 than expected, "
      f"compared to the league average (pooled over all seasons).")
show = rates[["league_gsax_60", "pool_n", "pool_gsax_60", "rate"]].rename(columns={
    "league_gsax_60": "league GSAx/60", "pool_n": "backups", "pool_gsax_60": "backup GSAx/60 (per-season rate)",
    "rate": "replacement rate (used)"})
show["goalie WAR (used)"] = gw.groupby("season")["war"].sum()
show["goalie WAR (per-season rate)"] = gw_per_season.groupby("season")["war"].sum()
show.index = show.index.map(season_label)
show.round(3)

1,702 goalie-seasons. Backups allow 0.275 more goals per 60 than expected, compared to the league average (pooled over all seasons).


,league GSAx/60,backups,backup GSAx/60 (per-season rate),replacement rate (used),goalie WAR (used),goalie WAR (per-season rate)
season,,,,,,
2008-09,-0.120,29,-0.502,-0.395,113.395,157.257
2009-10,-0.000,23,-0.109,-0.275,113.325,44.912
2010-11,-0.023,27,-0.157,-0.298,113.734,55.648
2011-12,0.001,29,-0.221,-0.275,113.856,91.689
2012-13,-0.041,23,-0.372,-0.317,66.528,79.978
2013-14,-0.015,37,-0.233,-0.290,113.741,89.966
2014-15,0.035,32,-0.251,-0.240,113.463,117.756
2015-16,0.068,32,-0.166,-0.207,113.150,96.194
2016-17,0.068,35,-0.676,-0.207,112.805,304.776


In [8]:
GOALIE_COLS = ["season", "season_label", "playerId", "name", "team", "games_played", "toi", "goals", "xGoals", "gsax",
               "gsax_60", "is_starter", "repl_rate", "gar", "war", "war_rank"]
assert not gw.duplicated(["playerId", "season"]).any()
gw[GOALIE_COLS].to_csv(OUT / "goalie_war_2008_2025.csv", index=False)
print(f"Saved {len(gw):,} goalie-seasons ({gw['season'].nunique()} seasons) to goalie_war_2008_2025.csv")

Saved 1,702 goalie-seasons (18 seasons) to goalie_war_2008_2025.csv


## 8. Validation: does adding goalies match team results better?
Same method as notebook 06: add up each team's WAR and correlate it with the team's 5-on-5 goal differential (sum of every skater's on-ice goals divided by 5). If goalie WAR captures something real, skaters + goalies should match better than skaters alone.

In [9]:
fit = team_check(war, gw, skaters_raw)
fit["teams"] = fit["teams"].astype(int)
fit["gain"] = fit["skaters + goalies"] - fit["skaters"]
fit.index = fit.index.map(season_label)
print("Seasons where adding goalies raised the correlation:", int((fit["gain"] > 0).sum()), "of", len(fit))
fit.loc["average"] = fit.mean()
assert fit.loc["average", "skaters + goalies"] > fit.loc["average", "skaters"]
fit.round(3)

Seasons where adding goalies raised the correlation: 18 of 18


,skaters,skaters + goalies,teams,gain
season,,,,
2008-09,0.783,0.828,30.000,0.045
2009-10,0.881,0.898,30.000,0.018
2010-11,0.741,0.816,30.000,0.075
2011-12,0.673,0.808,30.000,0.134
2012-13,0.634,0.803,30.000,0.168
2013-14,0.739,0.779,30.000,0.040
2014-15,0.814,0.874,30.000,0.059
2015-16,0.785,0.829,30.000,0.044
2016-17,0.863,0.917,30.000,0.053


## 9. Which goalie stats repeat?
Year-to-year correlation for the same goalie, back-to-back seasons, 1000+ minutes in both. Rates are per 60 minutes. Each stat is shown for all pairs and for goalies who **changed teams**, since a goalie who changes teams leaves his old defense behind.

### Team codes first
MoneyPuck's team codes changed over the years (`L.A` became `LAK` in 2021-22) and two franchises moved (Atlanta to Winnipeg, Arizona to Utah). `FRANCHISE` maps each old code to its franchise, so a code change doesn't count as a team change. The check fails if any code is unknown, or if an old code doesn't end the season before its new one starts.

In [10]:
codes = franchise_check(skaters_raw, goalies_raw)
print(f"{len(codes)} team codes, same in the skater and goalie files every season. FRANCHISE: {FRANCHISE}")
print("Codes missing from some seasons (1 = present; season = start year):")
print(codes[codes.eq(0).any(axis=1)].to_string())

pairs = season_pairs(goalies)
code_only = pairs[pairs["code_change_only"]]
assert not code_only["changed_team"].any()
assert all(FRANCHISE.get(a) == b for a, b in zip(code_only["team_1"], code_only["team_2"]))
print(f"\n{len(code_only)} pairs changed code but not franchise (not counted as a team change):")
print(code_only.assign(seasons=code_only["season"].map(lambda s: f"{season_label(s)} to {season_label(s + 1)}"))
               [["name_1", "seasons", "team_1", "team_2"]].rename(columns={"name_1": "goalie"}).to_string(index=False))
print(f"\nPairs: {len(pairs)}, changed team: {int(pairs['changed_team'].sum())}")

38 team codes, same in the skater and goalie files every season. FRANCHISE: {'L.A': 'LAK', 'N.J': 'NJD', 'S.J': 'SJS', 'T.B': 'TBL', 'ATL': 'WPG', 'ARI': 'UTA'}
Codes missing from some seasons (1 = present; season = start year):
season  2008  2009  2010  2011  2012  2013  2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025
team                                                                                                              
ARI        1     1     1     1     1     1     1     1     1     1     1     1     1     1     1     1     0     0
ATL        1     1     1     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0
L.A        1     1     1     1     1     1     1     1     1     1     1     1     1     0     0     0     0     0
LAK        0     0     0     0     0     0     0     0     0     0     0     0     0     1     1     1     1     1
N.J        1     1     1     1     1     1     1     1     1     1     1     1   

### Repeatability, raw and compared to each season's league average
The raw numbers can be inflated by something that has nothing to do with goalies: MoneyPuck's expected goals drift for **everyone** in some seasons. Two seasons in a row share that drift, so a goalie's numbers look "repeatable" even if he's average both years. The second set subtracts each season's league average first.

In [11]:
raw_r = repeatability(goalies)
adj_r = repeatability(goalies, adjust=True)
print(f"Pairs: {raw_r['all n'].iloc[0]} ({raw_r['changed team n'].iloc[0]} changed team); "
      f"4on5 row ({MIN_PK_TOI}+ PK minutes both seasons): {raw_r['all n'].iloc[-1]} ({raw_r['changed team n'].iloc[-1]})")
pd.concat({"raw": raw_r[["all r", "changed team r"]],
           "vs. season league average": adj_r[["all r", "changed team r"]]}, axis=1).round(2)

Pairs: 769 (179 changed team); 4on5 row (150+ PK minutes both seasons): 417 (72)


raw                vs. season league average               
                        all r changed team r                     all r changed team r
stat                                                                                 
GSAx                     0.13          -0.06                      0.12          -0.11
High-danger GSAx         0.47           0.37                      0.06          -0.10
Medium-danger GSAx       0.05          -0.12                      0.03          -0.14
Low-danger GSAx          0.22           0.10                      0.08          -0.02
Rebounds vs. expected    0.69           0.63                      0.29           0.09
Freezes vs. expected     0.72           0.67                      0.57           0.48
Save %                   0.30           0.13                      0.14          -0.08
5v5 GSAx                 0.09           0.01                      0.07          -0.05
4on5 GSAx (150+ PK min)  0.08          -0.03                      0.08          -0.04

### The drift: league-wide rates by season
From 2022-23 on, the whole league saves more high-danger goals than expected and gives up far fewer rebounds and freezes far fewer pucks than expected. That's the expected-goals model, not every goalie suddenly changing, and it's what inflates the raw high-danger and rebound numbers above. Goalie WAR isn't affected: its replacement rate moves with the league average each season.

In [12]:
by = goalies.groupby("season")
per60 = lambda more, less: (by[more].sum() - by[less].sum()) / by["toi"].sum() * 60
drift = pd.DataFrame({
    "GSAx/60": per60("xGoals", "goals"),
    "high-danger GSAx/60": per60("highDangerxGoals", "highDangerGoals"),
    "rebounds vs. exp./60": per60("xRebounds", "rebounds"),
    "freezes vs. exp./60": per60("freeze", "xFreeze"),
    "save %": 1 - by["goals"].sum() / by["ongoal"].sum(),
})
drift.index = drift.index.map(season_label)
drift.round(3)

,GSAx/60,high-danger GSAx/60,rebounds vs. exp./60,freezes vs. exp./60,save %
season,,,,,
2008-09,-0.120,-0.069,0.048,-0.058,0.909
2009-10,-0.000,-0.036,0.031,-0.107,0.911
2010-11,-0.023,-0.087,0.079,-0.024,0.913
2011-12,0.001,-0.026,-0.012,-0.102,0.914
2012-13,-0.041,-0.068,0.021,-0.011,0.912
2013-14,-0.015,-0.036,-0.006,0.126,0.914
2014-15,0.035,-0.031,0.019,0.374,0.915
2015-16,0.068,-0.015,-0.013,0.272,0.915
2016-17,0.068,0.007,-0.145,0.251,0.913


## 10. Leaders

In [13]:
LEAD = ["season_label", "name", "team", "games_played", "gsax", "gar", "war"]
print("Top 15 goalie WAR seasons, 2008-09 to 2025-26")
print(gw.nlargest(15, "war")[LEAD].round(1).to_string(index=False))
print("\n2025-26 leaders")
print(gw[gw["season"] == 2025].nlargest(10, "war")[LEAD[1:] + ["war_rank"]].round(1).to_string(index=False))

Top 15 goalie WAR seasons, 2008-09 to 2025-26
season_label               name team  games_played  gsax  gar  war
     2022-23        Juuse Saros  NSH            64  46.7 59.7 10.0
     2010-11         Tim Thomas  BOS            57  39.6 56.3  9.4
     2022-23      Linus Ullmark  BOS            49  42.4 52.3  8.7
     2024-25  Connor Hellebuyck  WPG            63  39.6 51.5  8.6
     2022-23       Ilya Sorokin  NYI            62  38.7 51.0  8.5
     2009-10   Henrik Lundqvist  NYR            73  31.6 50.8  8.5
     2021-22    Igor Shesterkin  NYR            53  34.1 50.7  8.4
     2011-12     Jonathan Quick  L.A            69  31.8 50.5  8.4
     2011-12         Mike Smith  ARI            67  32.3 50.2  8.4
     2014-15        Carey Price  MTL            66  33.2 49.1  8.2
     2021-22 Andrei Vasilevskiy  TBL            63  28.4 48.7  8.1
     2009-10        Ryan Miller  BUF            69  29.4 47.9  8.0
     2016-17      Braden Holtby  WSH            62  35.3 47.8  8.0
     2010-11    

## Takeaways
- **Skater WAR now covers 18 seasons** with no math changes. 2021-22 to 2025-26 are unchanged (section 3), and the leaders before that line up with hockey history: Ovechkin 2009-10, Malkin 2011-12, Crosby 2018-19, Pastrnak 2019-20. 2012-13, 2019-20 and 2020-21 were shortened, so their WAR totals are smaller.
- **Goalie WAR passes the team test.** Adding it raises the match with team goal differential in every season, from 0.78 to 0.85 on average.
- **A steady replacement level** (backups' gap below the league average, pooled over all seasons) keeps total goalie WAR stable from season to season. The per-season version swung from 27 to 305.
- **GSAx barely repeats** (0.12 vs. each season's league average, below zero for goalies who changed teams), so one great season says little about the next. That's a Forecast problem: goalie projections will need heavy regression to the mean.
- **The raw high-danger and rebound numbers are inflated** by league-wide drift in expected goals. Compared to each season's league average, high-danger GSAx doesn't repeat at all. **Freezing the puck** is the one goalie skill that clearly repeats and follows him to a new team.
- **Next:** goalie cards, and the Forecast's aging curves and regression to the mean.